# Phase 3 Cross-validation and Reward Inventory

This notebook reads Phase 2 interim CSV files, runs objective cross-validation without normalization, and writes `data/interim/reward_inventory.csv`. It does not create processed outputs.

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

interim_dir = project_root / 'data' / 'interim'
TODAY = pd.Timestamp('2026-06-10')

In [2]:
parsed_by_type = pd.read_csv(interim_dir / 'parsed_by_type.csv', encoding='utf-8')
parsed_by_date = pd.read_csv(interim_dir / 'parsed_by_date.csv', encoding='utf-8')
tidy_prenorm = pd.read_csv(interim_dir / 'tidy_prenorm.csv', encoding='utf-8')

for df in [parsed_by_type, parsed_by_date, tidy_prenorm]:
    df['date'] = pd.to_datetime(df['date'])

print('parsed_by_type rows:', len(parsed_by_type))
print('parsed_by_date rows:', len(parsed_by_date))
print('tidy_prenorm rows:', len(tidy_prenorm))

parsed_by_type rows: 1140
parsed_by_date rows: 1160
tidy_prenorm rows: 1160


## Cross-validation A: by_type stated cycle days

In [3]:
type_sorted = parsed_by_type.sort_values(['reward_raw', 'date']).copy()
type_sorted['actual_cycle_days'] = type_sorted.groupby('reward_raw')['date'].diff().dt.days
cycle_check = type_sorted[type_sorted['stated_cycle_days'].notna()].copy()
cycle_mismatch = cycle_check[
    cycle_check['actual_cycle_days'] != cycle_check['stated_cycle_days']
].copy()

print('cycle checked rows:', len(cycle_check))
print('cycle mismatch rows:', len(cycle_mismatch))
display(cycle_mismatch[[
    'reward_raw', 'date', 'stated_cycle_days', 'actual_cycle_days', 'trailing'
]].head(10))

cycle checked rows: 1041
cycle mismatch rows: 0


,reward_raw,date,stated_cycle_days,actual_cycle_days,trailing


## Cross-validation B: exact reward_raw frequency match

In [4]:
by_type_freq = parsed_by_type.groupby('reward_raw').size().rename('freq_by_type')
by_date_rewards = parsed_by_date[
    (~parsed_by_date['no_event_week']) & parsed_by_date['reward_raw'].notna()
].copy()
by_date_freq = by_date_rewards.groupby('reward_raw').size().rename('freq_by_date')

freq_compare = pd.concat([by_type_freq, by_date_freq], axis=1).fillna(0).astype(int).reset_index()
freq_compare['is_exact_count_match'] = freq_compare['freq_by_type'] == freq_compare['freq_by_date']
matched = int(freq_compare['is_exact_count_match'].sum())
total = len(freq_compare)
match_rate = matched / total if total else 0

print('exact reward unique total:', total)
print('exact count matched rewards:', matched)
print('exact count mismatched rewards:', total - matched)
print('exact count match rate:', match_rate)
display(freq_compare[~freq_compare['is_exact_count_match']]
        .sort_values(['freq_by_date', 'freq_by_type', 'reward_raw'], ascending=[False, False, True])
        .head(10))

exact reward unique total: 113
exact count matched rewards: 75
exact count mismatched rewards: 38
exact count match rate: 0.6637168141592921


,reward_raw,freq_by_type,freq_by_date,is_exact_count_match
63,스타포스 강화 비용 30% 할인,73,71,False
67,어빌리티 재설정 비용 50% 할인,69,68,False
86,주문의 흔적 강화 비용 50% 할인,68,67,False
62,"스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%",55,54,False
59,소울 조각 사용 시 위대한 소울 획득 확률 5배,53,52,False
1,10성 이하에서 스타포스 강화 성공 시 1+1 강화,42,41,False
95,폴로와 프리토의 현상금 사냥 경험치 2배,53,31,False
112,"폴로와 프리토, 에스페시아의 현상금 사냥 경험치 2배",0,22,False
107,사냥 시 소환되는 엘리트 몬스터 수 증가,0,19,False
41,미라클 타임,16,15,False


## Reward inventory

In [5]:
type_dates = parsed_by_type.groupby('reward_raw')['date'].agg(type_first='min', type_last='max')
date_dates = by_date_rewards.groupby('reward_raw')['date'].agg(date_first='min', date_last='max')
discontinued = parsed_by_type.groupby('reward_raw')['is_discontinued'].any().rename('is_discontinued')

inventory = freq_compare.set_index('reward_raw').join([type_dates, date_dates, discontinued])
inventory['first_date'] = inventory[['type_first', 'date_first']].min(axis=1)
inventory['last_date'] = inventory[['type_last', 'date_last']].max(axis=1)
inventory['is_discontinued'] = inventory['is_discontinued'].fillna(False).astype(bool)
inventory = inventory.reset_index()[[
    'reward_raw', 'freq_by_date', 'freq_by_type', 'first_date', 'last_date', 'is_discontinued'
]]
inventory = inventory.sort_values(
    ['freq_by_date', 'freq_by_type', 'reward_raw'], ascending=[False, False, True]
).reset_index(drop=True)
inventory['first_date'] = inventory['first_date'].dt.strftime('%Y-%m-%d')
inventory['last_date'] = inventory['last_date'].dt.strftime('%Y-%m-%d')
inventory.to_csv(interim_dir / 'reward_inventory.csv', index=False, encoding='utf-8')

print('reward_inventory rows:', len(inventory))
print('reward_inventory discontinued rewards:', int(inventory['is_discontinued'].sum()))
print('reward_inventory bytes:', (interim_dir / 'reward_inventory.csv').stat().st_size)
display(inventory.head(20))

reward_inventory rows: 113
reward_inventory discontinued rewards: 71
reward_inventory bytes: 9603


,reward_raw,freq_by_date,freq_by_type,first_date,last_date,is_discontinued
0,룬 경험치 버프 효과 +100%,87,87,2017-03-26,2026-06-07,False
1,콤보킬 경험치 획득량 +300%,86,86,2017-11-26,2026-06-07,False
2,스타포스 강화 비용 30% 할인,71,73,2017-04-23,2026-05-31,False
3,어빌리티 재설정 비용 50% 할인,68,69,2017-04-09,2026-05-17,False
4,주문의 흔적 강화 비용 50% 할인,67,68,2018-03-18,2026-04-19,False
5,"스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%",54,55,2017-03-26,2025-07-20,False
6,불꽃늑대 퇴장 시 획득하는 경험치 2배,53,53,2017-12-03,2024-08-18,True
7,소울 조각 사용 시 위대한 소울 획득 확률 5배,52,53,2017-04-23,2026-05-17,False
8,몬스터 파크 클리어 경험치 추가 250%,47,47,2023-12-24,2026-05-24,False
9,10성 이하에서 스타포스 강화 성공 시 1+1 강화,41,42,2018-02-18,2026-01-11,False


## Phase 3 parser correction checks

In [6]:
no_event = tidy_prenorm[tidy_prenorm['no_event_week']].copy()
empty_unknown = tidy_prenorm[
    (~tidy_prenorm['no_event_week']) & tidy_prenorm['reward_raw'].isna()
].copy()
future_empty = empty_unknown[empty_unknown['is_future']].copy()
past_empty = empty_unknown[~empty_unknown['is_future']].copy()

print('discontinued unique rewards in parsed_by_type:', parsed_by_type.loc[parsed_by_type['is_discontinued'], 'reward_raw'].nunique())
print('no_event_week rows:', len(no_event))
print('no_event_week future rows:', int(no_event['is_future'].sum()))
print('future empty blocks:', len(future_empty))
print('past empty blocks without sentinel:', len(past_empty))
print('date min:', tidy_prenorm['date'].min().date().isoformat())
print('date max:', tidy_prenorm['date'].max().date().isoformat())
print('unique dates:', tidy_prenorm['date'].nunique())
display(past_empty[['date', 'event_context', 'is_major_event', 'is_future', 'no_event_week']])

discontinued unique rewards in parsed_by_type: 71
no_event_week rows: 30
no_event_week future rows: 0
future empty blocks: 12
past empty blocks without sentinel: 8
date min: 2017-03-12
date max: 2026-08-30
unique dates: 501


,date,event_context,is_major_event,is_future,no_event_week
590,2023-01-12,요정 웡키의 웡스토랑 - 마이크의 파워업 위크,True,False,False
591,2023-01-12,(목) - 10성 이하에서 스타포스 강화 성공 시 1+1 강화,False,False,False
592,2023-01-13,(금) - 소울 조각 사용 시 위대한 소울 획득 확률 5배,False,False,False
593,2023-01-14,"토) - 스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%(샤이닝 스...",False,False,False
594,2023-01-15,일) - 미라클 타임(오전 8시~오후 11시 30분,False,False,False
595,2023-01-16,(월) - 주문의 흔적 강화 비용 50% 할인,False,False,False
596,2023-01-17,(화) - 어빌리티 재설정 비용 50% 할인,False,False,False
597,2023-01-18,(수) - 스타포스 강화 비용 30% 할인,False,False,False
